In [2]:
import pandas as pd
import re

df = pd.read_csv(
    "train.csv",
    engine="python",
    on_bad_lines="skip"
)

def clean_text_fast(text):
    if pd.isna(text):
        return text

    text = str(text)

    # 1. Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # 2. Remove all numbers
    # Doing this early prevents numbers from being treated as "words" or "characters"
    text = re.sub(r'\d+', '', text)

    # 3. Keep only letters, spaces, and specific punctuation
    # Removed '0-9' from the allowed list
    text = re.sub(r"[^a-zA-Z\s.,!?'\-]", '', text)

    # 4. ✅ Cap long runs of punctuation (!?.) to exactly 6
    # If there are 7 or more, it slices the first 6
    text = re.sub(r'([!?.])\1{6,}', r'\1\1\1\1\1\1', text)

    # 5. ✅ Collapse repeated letters to only ONE (e.g., "loooove" -> "love")
    # You mentioned limiting words/chars to one; this handles the character level.
    text = re.sub(r'([a-zA-Z])\1{2,}', r'\1\1', text)

# This looks for any sequence of 2+ characters that repeats itself
    # e.g., "one" "one" "one" -> "one"
    text = re.sub(r'(\w{2,})\1+', r'\1', text, flags=re.IGNORECASE)

    # 6. Fix "one one one" (Repeated words with spaces)
    text = re.sub(r"\b(\w+)(?:\s+\1\b)+", r"\1", text, flags=re.IGNORECASE)

    text = re.sub(r'(\b\w{2,})\1+', r'\1', text, flags=re.IGNORECASE)

    # 7. Normalize whitespace
    text = re.sub(r'\s+', ' ', text).strip()

    return text

# Apply and Save
df["comment_text"] = df["comment_text"].apply(clean_text_fast)
df.to_csv("train_cleaned_6.csv", index=False)

print("✅ Numbers removed, words/chars limited to 1, and punctuation capped at 6.")

✅ Numbers removed, words/chars limited to 1, and punctuation capped at 6.
